In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from utils import utils_ml 
from catboost import CatBoostRegressor
import glob 

In [ ]:
# Automatically find the matching file
input_files = glob.glob('training_features*.npy')
if not input_files:
    raise FileNotFoundError("No features_filtered*.npy file found in current directory")
input_path = input_files[0]

# Load memory-mapped array
print(f"Loading features from {input_path}")
X_arr = np.load(input_path, mmap_mode='r').astype(np.float32)

# Class mapping dictionary
class_to_poles = {
    0: [0, 0, 0],  # 1 pole on [bt]
    1: [1, 0, 0],  # 1 pole on [bb]
    2: [0, 1, 0],  # 1 pole on [tb]
    3: [0, 0, 1],  # 1 pole on [bt] and 1 pole on [bb]
    4: [2, 0, 0],  # 1 pole on [bb] and 1 pole on [tb]
    5: [0, 2, 0],  # 1 pole on each of [bt], [bb], and [tb]
    6: [0, 0, 2],  # 2 poles on [bb] and 1 pole on [tb]
    7: [1, 1, 0],  # 1 pole on [bb] and 2 poles on [tb]
    8: [1, 0, 1],
    9: [0, 1, 1],
    10: [3, 0, 0],
    11: [0, 3, 0],
    12: [0, 0, 3],
    13: [2, 1, 0],
    14: [2, 0, 1],
    15: [1, 2, 0],
    16: [0, 2, 1],
    17: [1, 0, 2],
    18: [0, 1, 2],
    19: [1, 1, 1],
    20: [4, 0, 0],
    21: [0, 4, 0],
    22: [0, 0, 4],
    23: [3, 1, 0],
    24: [3, 0, 1],
    25: [1, 3, 0],
    26: [0, 3, 1],
    27: [1, 0, 3],
    28: [0, 1, 3],
    29: [2, 2, 0],
    30: [2, 0, 2],
    31: [0, 2, 2],
    32: [2, 1, 1],
    33: [1, 2, 1],
    34: [1, 1, 2],
}


# File paths
file_prefix = "rawFeatures/P"
file_suffix = "_intensity.pkl"
num_files = 35

# Load labels
print("Loading labels...")
y_arr_classification = np.array([
    np.tile(i, 10000) 
    for i in np.arange(num_files)
]).flatten()

def convert_labels(old_labels, class_to_poles):
    """
    Convert class labels to pole representation.
    
    Parameters:
        old_labels (np.ndarray): Array of class labels.
        class_to_poles (dict): Mapping from class labels to pole representations.
        
    Returns:
        np.ndarray: A 2D array with pole representations.
    """
    # Initialize a new array with shape (length of old labels, 3)
    new_labels = np.zeros((len(old_labels), 3), dtype=int)
    
    # Populate the new label array based on the mapping
    for i, label in enumerate(old_labels):
        new_labels[i] = class_to_poles[label]
    
    return new_labels

indices = np.arange(350_000)
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42, shuffle=True)

y_arr_regression = convert_labels(y_arr_classification, class_to_poles) [test_idx]
X_arr = X_arr[test_idx]

Loading features from training_features_128_T11T12NoBg.npy
Loading labels...


In [4]:
r3_acc = []
for i in range(5):
    
    path_bb = f"models/catboost_model_r3_bb_fold{i}.cbm"
    path_bt = f"models/catboost_model_r3_bt_fold{i}.cbm"
    path_tb = f"models/catboost_model_r3_tb_fold{i}.cbm"

    model_bb = CatBoostRegressor()
    model_bt = CatBoostRegressor()
    model_tb = CatBoostRegressor()

    model_bb.load_model(path_bb)
    model_bt.load_model(path_bt)
    model_tb.load_model(path_tb)

    y_pred_bb = np.abs(np.round(model_bb.predict(X_arr)))
    y_pred_bt = np.abs(np.round(model_bt.predict(X_arr)))
    y_pred_tb = np.abs(np.round(model_tb.predict(X_arr)))

    y_pred = np.stack([y_pred_bt, y_pred_bb, y_pred_tb],axis=1)
    
    acc = accuracy_score(
        utils_ml.reconvert_labels(y_pred, class_to_poles),
        utils_ml.reconvert_labels(y_arr_regression, class_to_poles)
    )
    r3_acc.append(acc)